# Example 3.2 / 1: Check the Status Before Reading Values

Companion notebook for **3.2 Did It Solve? Solve Results** in the AMPL lab Documentation, section "Check the Status Before Reading Values". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check the Status Before Reading Values

This example deliberately has no feasible solution. The status is checked before any value of
`x` is used.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x;                     # Free decision
minimize Cost: x;          # Seek the smallest x
subject to Lower: x >= 2;  # Lower limit that Upper contradicts
subject to Upper: x <= 1;  # Conflicts with Lower: no feasible x

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
# This model has no external data.

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(int(ampl.get_value("solve_result_num")))
if ampl.solve_result == "solved":
    print(ampl.var["x"].value())
else:
    print("No solution values used")

infeasible
299
No solution values used


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
# This model has no external data.

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
solve;
display solve_result, solve_result_num;
if solve_result = "solved" then display x;
else print "No solution values used";
""")
except Exception:
    pass  # AMPL has printed the error above

[WARNING] 
	presolve, variable x:
		impossible deduced bounds: lower = 2, upper = 1;
		difference = 1


solve_result = infeasible
solve_result_num = 299

No solution values used
